In [5]:
import pandas as pd
import numpy as np

file_path = '时间对应等级.xlsx'
df = pd.read_excel(file_path, na_values=['未分类', 'NaN', 'nan', ''])
df_clean = df.dropna().copy()
for col in df_clean.columns:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
df_clean = df_clean.dropna()

days = df_clean.iloc[:, 0].values
model1 = df_clean.iloc[:, 1].values
model2 = df_clean.iloc[:, 2].values
model3 = df_clean.iloc[:, 3].values

def analyze_time_series_early_exempt(grades, days):
    n = len(grades)
    diffs = np.diff(grades)
    
    is_backward = np.zeros(n-1, dtype=bool)
    is_exempt_early = np.zeros(n-1, dtype=bool)
    
    for i in range(n-1):
        prev_val = grades[i]
        curr_val = grades[i+1]
        current_day = days[i+1]
        
        if curr_val < prev_val:
            if current_day <= 20:
                is_exempt_early[i] = True
            else:
                is_backward[i] = True
    
    total_possible = n - 1
    backward_count = np.sum(is_backward)
    exempt_early_count = np.sum(is_exempt_early)
    
    effective_events = total_possible - exempt_early_count
    monotonicity_score = 1 - backward_count / effective_events if effective_events > 0 else 1.0
    
    return monotonicity_score

print("="*50)
print("Monotonicity Score")
print("="*50)

model_names = ['Model 1', 'Model 2', 'Model 3']
model_data = [model1, model2, model3]

for name, data in zip(model_names, model_data):
    score = analyze_time_series_early_exempt(data, days)
    print(f"\n[{name}] Monotonicity Score: {score:.4f}")

print("\n" + "="*50)

Monotonicity Score

[Model 1] Monotonicity Score: 0.8695

[Model 2] Monotonicity Score: 0.8541

[Model 3] Monotonicity Score: 0.8995



In [4]:
import pandas as pd
import numpy as np

pd.set_option('future.no_silent_downcasting', True)

EXCEL_PATH = r"时间对应等级.xlsx"
LEVELS = [-4, -3, -2, -1, 1, 2]

def time_weighted_median(series, level):
    if series.empty:
        return np.nan
    
    sorted_vals = series.sort_values()
    days = sorted_vals.values
    n = len(days)

    if level == -4:
        weights = np.linspace(10, 0.1, n)
    elif level == -3:
        weights = np.linspace(1, 2, n)
    elif level == -2:
        weights = np.linspace(1, 2.5, n)
    elif level == -1:
        weights = np.linspace(2, 1, n)
    elif level == 1:
        weights = np.linspace(1, 0.3, n)
    elif level == 2:
        weights = np.linspace(1, 1, n)

    total_weight = weights.sum()
    cumulative = 0
    half = total_weight / 2
    for val, w in zip(sorted_vals, weights):
        cumulative += w
        if cumulative >= half:
            return val
    return sorted_vals.iloc[-1]

df = pd.read_excel(EXCEL_PATH)
day_col = df.columns[0]
model_cols = df.columns[1:4]

df_clean = df.copy()
for col in model_cols:
    df_clean[col] = df_clean[col].replace("未分类", pd.NA)

result = []
for model in model_cols:
    for level in LEVELS:
        filtered = df_clean[df_clean[model] == level]
        if not filtered.empty:
            median_day = int(time_weighted_median(filtered[day_col], level))
        else:
            median_day = "No Corresponding Days"
        result.append({"Model": "Model", "Level": level, "Median Compost Days": median_day})

result_df = pd.DataFrame(result)
print(result_df)
result_df.to_excel("Median Days Result by Level.xlsx", index=False)

    Model  Level  Median Compost Days
0   Model     -4                    5
1   Model     -3                   20
2   Model     -2                    9
3   Model     -1                    5
4   Model      1                   63
5   Model      2                   30
6   Model     -4                   15
7   Model     -3                   12
8   Model     -2                    9
9   Model     -1                    5
10  Model      1                   49
11  Model      2                   34
12  Model     -4                    5
13  Model     -3                    8
14  Model     -2                   13
15  Model     -1                   17
16  Model      1                   41
17  Model      2                   55
